In [1]:
# Global Top Words Extraction for All Models
# This notebook extracts the global top words for the best-tuned models of LDA, BERTopic, Top2Vec, TopicGPT, and DTM.
# The results are saved to results/{model}/modeling/global_top_words.csv

In [1]:
import os
import gc
import pickle
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")

# Define Data Paths
BASE_DIR = Path("../../data")
MODELS_DIR = Path("../../models")
RESULTS_DIR = Path("../../results")
LIST_SUBJECT = ["cs", "math", "physics"]
# Ensure directories exist
for model in ["lda", "bertopic", "top2vec", "topicGpt", "dtm"]:
    (RESULTS_DIR / model / "modeling").mkdir(parents=True, exist_ok=True)


In [3]:
# 1. LDA
def extract_lda_global_top_words():
    print("Extracting LDA global top words...")
    rows = []
    for subject in LIST_SUBJECT:
        try:
            model_path = MODELS_DIR / "lda" / "tuning" / subject / "best_model.pkl"
            if not model_path.exists():
                print(f"  Missing LDA model for {subject}")
                continue
            
            with open(model_path, "rb") as f:
                lda_model = pickle.load(f)
            
            for topic_id in range(lda_model.num_topics):
                # get top 20 words globally
                words = [w for w, _ in lda_model.show_topic(topic_id, topn=20)]
                rows.append({"subject": subject, "topic_id": topic_id, "top_words": ", ".join(words)})
            print(f"  Done for {subject}")
        except Exception as e:
            print(f"  Error {subject}: {e}")
            
    if rows:
        df = pd.DataFrame(rows)
        df.to_csv(RESULTS_DIR / "lda" / "modeling" / "global_top_words.csv", index=False)
        print("Saved LDA top words")

extract_lda_global_top_words()

Extracting LDA global top words...
  Done for cs
  Done for math
  Done for physics
Saved LDA top words


In [2]:
def extract_bertopic_global_top_words(n_words=20):
    print("Extracting BERTopic global top words...")
    from bertopic import BERTopic
    rows = []
    for subject in LIST_SUBJECT:
        try:
            model_path = MODELS_DIR / "bertopic" / "tuning" / "hdbscan" / f"best_{subject}_quality"
            if getattr(model_path, "exists", lambda: os.path.exists(model_path))():
                model = BERTopic.load(str(model_path))

                emb_path = BASE_DIR / "preprocess" / subject / "emb" / "v1.csv"
                
                docs_df = pd.read_csv(emb_path)
                docs = docs_df["text"].dropna().astype(str).tolist()  

                model.update_topics(docs=docs, top_n_words=n_words)

                topic_info = model.get_topic_info()
                
                for _, row in topic_info.iterrows():
                    topic_id = row["Topic"]
                    if topic_id == -1:
                        continue

                    topic_words = model.get_topic(topic_id)
                    words = [word for word, _ in topic_words[:n_words]]

                    rows.append({
                        "subject": subject,
                        "topic_id": topic_id,
                        "top_words": ", ".join(words)
                    })

                print(f"  Done for {subject}")
                del model
                gc.collect()
        except Exception as e:
            print(f"  Error {subject}: {e}")
    
    if rows:
        df = pd.DataFrame(rows)
        df.to_csv(RESULTS_DIR / "bertopic" / "modeling" / "global_top_words.csv", index=False)
        print("Saved BERTopic top words")

extract_bertopic_global_top_words(n_words=20)

Extracting BERTopic global top words...
  Done for cs
  Done for math
  Done for physics
Saved BERTopic top words


In [5]:
# 3. Top2Vec
def extract_top2vec_global_top_words():
    print("Extracting Top2Vec global top words...")
    from top2vec import Top2Vec
    rows = []
    for subject in LIST_SUBJECT:
        try:
            # try tuning directory
            model_path = MODELS_DIR / "top2vec" / "tuning" / subject / "model"
            if not model_path.exists():
                print(f"  Missing Top2Vec model for {subject}")
                continue
                
            model = Top2Vec.load(str(model_path))
            topic_words, _, topic_nums = model.get_topics(model.get_num_topics())
            
            for t_num, words in zip(topic_nums, topic_words):
                rows.append({"subject": subject, "topic_id": t_num, "top_words": ", ".join(words[:20])})
            print(f"  Done for {subject}")
            del model
            gc.collect()
        except Exception as e:
            print(f"  Error {subject}: {e}")
            
    if rows:
        df = pd.DataFrame(rows)
        df.to_csv(RESULTS_DIR / "top2vec" / "modeling" / "global_top_words.csv", index=False)
        print("Saved Top2Vec top words")

extract_top2vec_global_top_words()

Extracting Top2Vec global top words...
  Done for cs
  Done for math
  Done for physics
Saved Top2Vec top words


In [2]:
# 4. DTM
def extract_dtm_global_top_words():
    print("Extracting DTM global top words...")
    import tomotopy as tp
    rows = []
    for subject in LIST_SUBJECT:
        try:
            model_path = MODELS_DIR / "dtm" / "tuning" / subject / "best_model.bin"
            if not model_path.exists():
                print(f"  Missing DTM model for {subject}")
                continue
                
            model = tp.DTModel.load(str(model_path))
            
            for t in range(model.k):
                words = []
                for tp_idx in range(model.num_timepoints):
                    words.extend([w for w, _ in model.get_topic_words(t, timepoint=tp_idx, top_n=10)])
                
                # Aggregate to global top words based on frequency or just use the whole vocab distribution if possible
                # Tomotopy DTM also allows get_topic_words without timepoint in some versions, but if not, count frequencies:
                from collections import Counter
                word_counts = Counter(words)
                top_words = [w for w, _ in word_counts.most_common(20)]
                
                rows.append({"subject": subject, "topic_id": t, "top_words": ", ".join(top_words)})
            print(f"  Done for {subject}")
            del model
            gc.collect()
        except Exception as e:
            print(f"  Error {subject}: {e}")
            
    if rows:
        df = pd.DataFrame(rows)
        df.to_csv(RESULTS_DIR / "dtm" / "modeling" / "global_top_words.csv", index=False)
        print("Saved DTM top words")

extract_dtm_global_top_words()

Extracting DTM global top words...
  Done for cs
  Done for math
  Done for physics
Saved DTM top words


In [2]:
# 5. TopicGPT
def extract_topicgpt_global_top_words():
    print("Extracting TopicGPT global top words via c-TF-IDF...")
    from sklearn.feature_extraction.text import CountVectorizer
    rows = []
    for subject in LIST_SUBJECT:
        try:
            ckpt_path = MODELS_DIR / "topicGpt" / subject / "tuning_best.pkl"
            if not ckpt_path.exists():
                print(f"  Missing TopicGPT model for {subject}")
                continue
                
            with open(ckpt_path, "rb") as f:
                ckpt = pickle.load(f)
            
            df_assign = ckpt["assignment_df"]
            
            # Load Full text from preprocess
            try:
                emb_path = BASE_DIR / "preprocess" / subject / "emb" / "v1.csv"
                if not emb_path.exists():
                    print(f"  Missing preprocess v1.csv for {subject}")
                    continue
                df_text = pd.read_csv(emb_path)
            except Exception as e:
                 print(f"  Error loading preprocess text: {e}")
                 continue

            df_assign["text"] = df_assign["doc_idx"].map(df_text["text"].fillna(""))
            
            # Group by topic
            topic_docs = df_assign.groupby("topic_id")["text"].apply(" ".join).reset_index()
            
            vectorizer = CountVectorizer(stop_words="english")
            tf_matrix = vectorizer.fit_transform(topic_docs["text"])
            vocab = vectorizer.get_feature_names_out()
            
            # IDF computing
            n_groups = tf_matrix.shape[0]
            df_t = (tf_matrix > 0).sum(axis=0).A1 
            idf = np.log((n_groups + 1) / (df_t + 1)) + 1 
            tfidf_matrix = tf_matrix.multiply(idf).toarray()
            
            # Normalise
            row_sums = tfidf_matrix.sum(axis=1, keepdims=True)
            row_sums[row_sums == 0] = 1
            tfidf_matrix = tfidf_matrix / row_sums
            
            for idx, row in topic_docs.iterrows():
                t_id = row["topic_id"]
                scores = tfidf_matrix[idx]
                top_indices = scores.argsort()[-20:][::-1]
                words = [vocab[i] for i in top_indices if scores[i] > 0]
                rows.append({"subject": subject, "topic_id": t_id, "top_words": ", ".join(words)})
            print(f"  Done for {subject}")
        except Exception as e:
            print(f"  Error {subject}: {e}")
            
    if rows:
        df = pd.DataFrame(rows)
        df.to_csv(RESULTS_DIR / "topicGpt" / "modeling" / "global_top_words.csv", index=False)
        print("Saved TopicGPT top words")

extract_topicgpt_global_top_words()

Extracting TopicGPT global top words via c-TF-IDF...
  Done for cs
  Done for math
  Done for physics
Saved TopicGPT top words
